# 碳酸锂期货数据获取脚本

本脚本用于获取和清洗碳酸锂期货数据。

## 数据来源说明

由于公共API限制，数据获取需要手动从以下网站下载CSV：
1. **广州期货交易所** (gfex.com.cn) — 合约规则、交割数据
2. **东方财富期货** (quote.eastmoney.com/qihuo/lcm.html) — 日度价格数据
3. **上海有色网 SMM** (smm.cn) — 现货价格数据

## 数据格式

期望的CSV格式：
- `date`: 交易日期 (YYYY-MM-DD)
- `open`: 开盘价
- `high`: 最高价
- `low`: 最低价
- `close`: 收盘价
- `volume`: 成交量（手）
- `open_interest`: 持仓量（手）

In [ ]:
# ============================================
# 数据加载与预处理
# ============================================
import pandas as pd
import numpy as np
from pathlib import Path

# 设置路径
DATA_DIR = Path('./')

In [ ]:
# 加载原始数据
# 请将下载的CSV文件放在 data/ 目录下

def load_and_clean_data(filepath):
    """
    加载并清洗碳酸锂期货日度数据
    
    参数:
        filepath: CSV文件路径
    返回:
        清洗后的DataFrame
    """
    df = pd.read_csv(filepath)
    
    # 统一列名
    column_map = {
        '日期': 'date', '开盘': 'open', '最高': 'high',
        '最低': 'low', '收盘': 'close', '成交量': 'volume',
        '持仓量': 'open_interest', '结算价': 'settlement',
        '涨跌幅': 'change_pct'
    }
    df = df.rename(columns={k: v for k, v in column_map.items() if k in df.columns})
    
    # 日期格式转换
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
    
    # 价格单位统一为万元/吨
    price_cols = ['open', 'high', 'low', 'close', 'settlement']
    for col in price_cols:
        if col in df.columns:
            # 如果价格数值较大（元/吨），转换为万元/吨
            if df[col].max() > 50000:
                df[col] = df[col] / 10000
    
    return df

print("数据加载函数已定义")

In [ ]:
# 计算技术指标
def calculate_indicators(df):
    """计算常用技术指标"""
    
    if 'close' not in df.columns:
        print("缺少收盘价数据")
        return df
    
    # 日收益率
    df['return'] = df['close'].pct_change()
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    
    # 移动平均线
    df['MA5'] = df['close'].rolling(5).mean()
    df['MA10'] = df['close'].rolling(10).mean()
    df['MA20'] = df['close'].rolling(20).mean()
    df['MA60'] = df['close'].rolling(60).mean()
    
    # 波动率（20日滚动年化波动率）
    df['volatility_20d'] = df['log_return'].rolling(20).std() * np.sqrt(252)
    
    # 成交量移动平均
    if 'volume' in df.columns:
        df['volume_MA5'] = df['volume'].rolling(5).mean()
        df['volume_MA20'] = df['volume'].rolling(20).mean()
    
    return df

print("技术指标函数已定义")

In [ ]:
# 使用示例
# 将下载的数据文件放到 data/ 目录后，取消注释以下代码运行

# df = load_and_clean_data('LC_price_data.csv')
# df = calculate_indicators(df)
# print(f"数据范围: {df['date'].min()} 至 {df['date'].max()}")
# print(f"总交易日: {len(df)}")
# print(df.tail())

print("\n数据获取脚本准备就绪。")
print("请从以下网站手动下载数据：")
print("1. https://quote.eastmoney.com/qihuo/lcm.html")
print("2. https://www.gfex.com.cn/gfex/tsl/sspz.shtml")